# Reproduce Report Results

This is the submission-facing review/reproduction notebook for the code repository. It supersedes the earlier two-notebook cleanup attempt and is intended to be the first file a marker or group member opens.

In default review mode it loads and displays saved report-ready outputs where available, runs a quick COS/Carr--Madan validation, and prints actionable messages for any missing required outputs.


## Runtime note

Long neural-network training runs are disabled by default. The notebook is still useful in review mode because it displays saved tables and figures when they exist and tells the reader exactly how to regenerate missing outputs. Switch the run flags below only when a fresh computational rerun is intended.


## Recommended Colab runtime

Use a GPU runtime for full generation:

Runtime -> Change runtime type -> T4 GPU or better.

The notebook saves outputs to Google Drive after each major section when running in Colab with `SAVE_TO_DRIVE = True`. Progress is logged to `logs/run_progress.csv`. This is important because long Heston or multi-seed training runs can be interrupted by Colab runtime limits.


## Project setup and imports

Purpose:
Set up dependencies, paths, run flags, and clean output folders.

Report use:
Provides shared infrastructure for every section below.

Key caveat:
This notebook does not pretend that expensive training is instant; long reruns are routed to the original source notebooks unless a short validation can be run directly.

Outputs:
Creates the `results/report/` and `figures/report/` folder tree.


In [ ]:
from pathlib import Path
import json
import shutil
import sys
import os
import zipfile
from datetime import datetime, timezone

try:
    import pandas as pd
except ImportError as exc:
    raise ImportError("This notebook requires pandas. Install dependencies with `pip install -r requirements.txt`.") from exc

try:
    from IPython.display import display, Image, Markdown
except ImportError as exc:
    raise ImportError("This notebook is intended for Jupyter/IPython. Install dependencies with `pip install -r requirements.txt`.") from exc

def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    markers = [".git", "README.md", "requirements.txt"]
    for candidate in [start, *start.parents]:
        if any((candidate / marker).exists() for marker in markers):
            return candidate
    return start

PROJECT_ROOT = find_project_root()
RESULTS_ROOT = PROJECT_ROOT / "results" / "report"
FIGURES_ROOT = PROJECT_ROOT / "figures" / "report"
LOGS_ROOT = PROJECT_ROOT / "logs"
ARCHIVE_NOTEBOOKS = PROJECT_ROOT / "notebooks" / "archive" / "original_notebooks"

RUN_MODE = "review"  # "review" or "full_generation"
RUN_TRAINING = False  # full generation is handled by the dispatcher below
RUN_LONG_MULTI_SEED = False
RUN_HESTON_TRAINING = False  # full generation is handled by the dispatcher below
RUN_HESTON_FULL_GENERATION = False
RUN_HESTON_FULLINFO = True
RUN_HESTON_OBSVOL = True
RUN_HESTON_MULTI_SEED = False
RUN_QUICK_VALIDATION = True
LOAD_EXISTING_OUTPUTS = True
DOWNLOAD_ZIP_AT_END = False

INCLUDE_DATA_GENERATION = True
INCLUDE_TRANSACTION_COSTS = True
INCLUDE_OBSERVABLE_VOL_HESTON = True
INCLUDE_CVAR_TRAINED_LOSS = False

# Colab / Google Drive persistence. Outside Colab this remains harmless and all
# outputs stay in the local repository tree.
try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

SAVE_TO_DRIVE = True
DRIVE_SUBDIR = "MFE_neural_hedging_report_outputs"

if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_SUBDIR
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Saving persistent outputs to:", DRIVE_ROOT)
else:
    DRIVE_ROOT = None
    print("Google Drive saving disabled or not running in Colab.")

LOCAL_OUTPUT_ROOT = PROJECT_ROOT
DRIVE_OUTPUT_ROOT = DRIVE_ROOT

RESULTS = {
    "black_scholes": RESULTS_ROOT / "black_scholes",
    "data_generation": RESULTS_ROOT / "data_generation",
    "robustness": RESULTS_ROOT / "robustness",
    "parameter_conditioned": RESULTS_ROOT / "parameter_conditioned",
    "transaction_costs": RESULTS_ROOT / "transaction_costs",
    "heston_fullinfo": RESULTS_ROOT / "heston_fullinfo",
    "heston_obsvol": RESULTS_ROOT / "heston_obsvol",
    "validation": RESULTS_ROOT / "validation",
}
FIGURES = {name: FIGURES_ROOT / name for name in RESULTS}

for path in list(RESULTS.values()) + list(FIGURES.values()) + [LOGS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)
if DRIVE_ROOT is not None:
    for rel in ["results/report", "figures/report", "logs", "zips"]:
        (DRIVE_ROOT / rel).mkdir(parents=True, exist_ok=True)

SOURCE_NOTEBOOKS = {
    "black_scholes_and_parameter_conditioned": PROJECT_ROOT / "final_benchmark_comparison.ipynb",
    "architecture_selection": PROJECT_ROOT / "ideal_minimal_task_hyperparameter_tuning_with_architectures.ipynb",
    "parameter_robustness": PROJECT_ROOT / "parameter_robustness_study.ipynb",
    "transaction_costs": PROJECT_ROOT / "transaction_cost_neural_hedging_extension (1).ipynb",
    "heston_corrected": PROJECT_ROOT / "heston_stock_option_neural_hedging_COS_obsvol_multiseed_corrected.ipynb",
}

HESTON_MODES = [
    {
        "name": "fullinfo",
        "use_observable_vol": False,
        "description": "Full-information Heston run using true simulated variance v_t.",
        "results_dir": RESULTS["heston_fullinfo"],
        "figures_dir": FIGURES["heston_fullinfo"],
    },
    {
        "name": "obsvol",
        "use_observable_vol": True,
        "description": "Observable-volatility robustness run using an EWMA proxy for v_t.",
        "results_dir": RESULTS["heston_obsvol"],
        "figures_dir": FIGURES["heston_obsvol"],
    },
]

print("PROJECT_ROOT:", PROJECT_ROOT)
print(f"Default review mode: LOAD_EXISTING_OUTPUTS={LOAD_EXISTING_OUTPUTS}, RUN_TRAINING={RUN_TRAINING}, RUN_HESTON_TRAINING={RUN_HESTON_TRAINING}")


## Display helpers and long-run hooks

Purpose:
Load and display existing report outputs, display figures, and give honest regeneration routes for missing outputs.

Report use:
Turns the notebook from a path checklist into a review notebook.

Key caveat:
Missing files are not fabricated. They are reported with the expected path and regeneration route.

Outputs:
No report outputs directly.


In [ ]:
def now_utc():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def ensure_parent(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def drive_path_for(local_path):
    local_path = Path(local_path)
    if DRIVE_ROOT is None:
        return None
    try:
        relative = local_path.relative_to(PROJECT_ROOT)
    except ValueError:
        relative = Path(local_path.name)
    return DRIVE_ROOT / relative


def mirror_to_drive(local_path):
    local_path = Path(local_path)
    if DRIVE_ROOT is None or not local_path.exists():
        return None
    drive_path = drive_path_for(local_path)
    drive_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(local_path, drive_path)
    print(f"Copied to Drive: {drive_path}")
    return drive_path


def log_progress(section, status, local_output=None, drive_output=None, notes=""):
    row = pd.DataFrame([{
        "timestamp": now_utc(),
        "section": section,
        "status": status,
        "local_output": str(local_output) if local_output else "",
        "drive_output": str(drive_output) if drive_output else "",
        "notes": notes,
    }])
    progress_path = LOGS_ROOT / "run_progress.csv"
    progress_path.parent.mkdir(parents=True, exist_ok=True)
    if progress_path.exists():
        existing = pd.read_csv(progress_path)
        out = pd.concat([existing, row], ignore_index=True)
    else:
        out = row
    out.to_csv(progress_path, index=False)
    mirror_to_drive(progress_path)
    return progress_path


def save_dataframe(df, local_path, index=False, section="", notes=""):
    local_path = ensure_parent(local_path)
    df.to_csv(local_path, index=index)
    print(f"Saved CSV: {local_path}")
    drive_path = mirror_to_drive(local_path)
    log_progress(section or "save_dataframe", "saved", local_path, drive_path, notes)
    return local_path


def save_json(obj, local_path, section="", notes=""):
    local_path = ensure_parent(local_path)
    local_path.write_text(json.dumps(obj, indent=2))
    print(f"Saved JSON: {local_path}")
    drive_path = mirror_to_drive(local_path)
    log_progress(section or "save_json", "saved", local_path, drive_path, notes)
    return local_path


def save_text(text, local_path, section="", notes=""):
    local_path = ensure_parent(local_path)
    local_path.write_text(text)
    print(f"Saved text: {local_path}")
    drive_path = mirror_to_drive(local_path)
    log_progress(section or "save_text", "saved", local_path, drive_path, notes)
    return local_path


def save_figure(fig, local_path, dpi=200, bbox_inches="tight", section="", notes=""):
    local_path = ensure_parent(local_path)
    fig.savefig(local_path, dpi=dpi, bbox_inches=bbox_inches)
    print(f"Saved figure: {local_path}")
    drive_path = mirror_to_drive(local_path)
    log_progress(section or "save_figure", "saved", local_path, drive_path, notes)
    return local_path


def display_csv(path, title=None, max_rows=30, used_for=None, regenerate=None):
    path = Path(path)
    section = title or path.stem
    if title:
        display(Markdown(f"### {title}"))
    if path.exists():
        df = pd.read_csv(path)
        if len(df) > max_rows:
            display(df.head(max_rows))
            print(f"Displayed first {max_rows} of {len(df)} rows from {path.relative_to(PROJECT_ROOT)}")
        else:
            display(df)
        log_progress(section, "displayed", path, drive_path_for(path), used_for or "")
        return df
    print("MISSING REQUIRED OUTPUT:")
    print(f"Expected file: {path.relative_to(PROJECT_ROOT)}")
    if used_for:
        print(f"Used for: {used_for}")
    if regenerate:
        print(f"How to regenerate: {regenerate}")
    log_progress(section, "missing", path, drive_path_for(path), regenerate or "")
    return None


def display_figure(path, title=None, used_for=None, regenerate=None):
    path = Path(path)
    section = title or path.stem
    if title:
        display(Markdown(f"### {title}"))
    if path.exists():
        display(Image(filename=str(path)))
        log_progress(section, "displayed", path, drive_path_for(path), used_for or "")
        return True
    print("MISSING REQUIRED OUTPUT:")
    print(f"Expected file: {path.relative_to(PROJECT_ROOT)}")
    if used_for:
        print(f"Used for: {used_for}")
    if regenerate:
        print(f"How to regenerate: {regenerate}")
    log_progress(section, "missing", path, drive_path_for(path), regenerate or "")
    return False


def long_run_hook(flag_enabled, route, section="long-run hook"):
    # This keeps default review mode runnable in Colab while being honest about
    # what still lives in the source notebooks. The full training code is not
    # silently simulated here.
    if flag_enabled:
        print("LONG RUN REQUESTED:")
        print(route)
        print("This notebook records the route and continues; connect this hook to a short callable before expecting automatic retraining.")
        log_progress(section, "long_run_requested", notes=route)
    else:
        print(f"Long rerun disabled. Regeneration route: {route}")
        log_progress(section, "long_run_disabled", notes=route)


def copy_if_present(src, dest, section="copy output"):
    src, dest = Path(src), Path(dest)
    if not src.exists():
        return False
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dest)
    drive_path = mirror_to_drive(dest)
    log_progress(section, "copied", dest, drive_path, f"source={src}")
    print(f"Copied {src} -> {dest.relative_to(PROJECT_ROOT)}")
    return True


def heston_metadata(mode):
    return {
        "mode": mode["name"],
        "use_observable_vol": mode["use_observable_vol"],
        "description": mode["description"],
        "S0": 1.0,
        "K": 0.9,
        "T": 0.5,
        "r": 0.0,
        "N": 125,
        "v0": 0.16,
        "kappa": 2.0,
        "theta": 0.16,
        "xi": 0.60,
        "rho": -0.70,
        "Kh": 1.0,
        "Th": 1.0,
        "cos_terms": 256,
        "cos_truncation": [-7, 7],
        "timestamp": now_utc(),
    }


def write_heston_mode_metadata(mode):
    mode_name = mode["name"]
    out = mode["results_dir"] / f"heston_run_metadata_{mode_name}.json"
    return save_json(heston_metadata(mode), out, section=f"Heston metadata {mode_name}")


def run_heston_experiment(mode_name, use_observable_vol):
    mode = next(m for m in HESTON_MODES if m["name"] == mode_name)
    write_heston_mode_metadata(mode)
    route = (
        "run heston_stock_option_neural_hedging_COS_obsvol_multiseed_corrected.ipynb "
        f"with USE_OBSERVABLE_VOL={use_observable_vol} and copy mode-suffixed outputs here"
    )
    long_run_hook(RUN_HESTON_TRAINING, route, section=f"Heston {mode_name} generation")
    return route


## Full-generation dispatcher

Purpose:
Run the callable report generators when `RUN_MODE = "full_generation"`.

Report use:
Creates the clean CSVs, figures, Heston mode outputs, validation files, metadata, logs, and Drive mirrors needed for code submission.

Key caveat:
Some expensive neural-network rows are regenerated as explicitly labelled reported-run outputs rather than retrained. This is recorded in the CSV `Source` columns and generation notes.

Outputs:
Populates `results/report/` and `figures/report/`, then mirrors generated files to Drive when enabled.


In [ ]:
generated_in_current_run = set()

if RUN_MODE == "full_generation":
    display(Markdown("### Running full-generation output pass"))
    from src.report_generation import generate_all_report_outputs

    generation = generate_all_report_outputs(
        RESULTS_ROOT,
        FIGURES_ROOT,
        include_transaction_costs=INCLUDE_TRANSACTION_COSTS,
        include_obsvol=INCLUDE_OBSERVABLE_VOL_HESTON,
    )
    for local_path in generation.local_paths:
        generated_in_current_run.add(str(Path(local_path).relative_to(PROJECT_ROOT)))
        drive_path = mirror_to_drive(local_path)
        log_progress("full_generation", "generated", local_path, drive_path, "callable src generator")
    if generation.notes:
        notes_path = RESULTS_ROOT / "generation_notes.txt"
        save_text("\n".join(generation.notes) + "\n", notes_path, section="full_generation notes")
    print(f"Generated {len(generated_in_current_run)} report output files.")
else:
    print("Review mode: existing outputs will be loaded/displayed where present. Set RUN_MODE='full_generation' to regenerate clean outputs.")


## LaTeX filename mapping

Purpose:
Map figure/table names that appear in the LaTeX report to clean repository paths.

Report use:
Helps a reviewer connect the code bundle to the report's tables and figures.

Key caveat:
Some LaTeX paths reflect older output folders. The clean repo path is the submission-facing location where verified outputs should be copied.

Outputs:
Displayed mapping table; same mapping is mirrored in `docs/results_manifest.md`.


In [ ]:
latex_mapping = pd.DataFrame([
    {"Report table/figure": "Final Black--Scholes benchmark table", "LaTeX filename/path": "final_benchmark_metrics.csv", "Clean repo path": "results/report/black_scholes/final_benchmark_metrics.csv", "Generated by": "final_benchmark_comparison.ipynb", "Notes": "Clean path standardises the output location."},
    {"Report table/figure": "Four-strategy P&L distribution", "LaTeX filename/path": "pnl_four_strategies.png", "Clean repo path": "figures/report/black_scholes/pnl_four_strategies.png", "Generated by": "final_benchmark_comparison.ipynb", "Notes": "Use LaTeX filename when copying into clean folder."},
    {"Report table/figure": "Left-tail P&L zoom", "LaTeX filename/path": "pnl_left_tail_zoom.png", "Clean repo path": "figures/report/black_scholes/pnl_left_tail_zoom.png", "Generated by": "final_benchmark_comparison.ipynb", "Notes": "Expected from final benchmark cells."},
    {"Report table/figure": "Average delta by moneyness", "LaTeX filename/path": "final_benchmark_outputs/average_delta_by_moneyness.png", "Clean repo path": "figures/report/black_scholes/average_delta_by_moneyness.png", "Generated by": "final_benchmark_comparison.ipynb", "Notes": "Clean folder removes old output ambiguity."},
    {"Report table/figure": "Robustness RMSE vs N", "LaTeX filename/path": "parameter_robustness_outputs/robustness_rmse_vs_N.png", "Clean repo path": "figures/report/robustness/robustness_rmse_vs_N.png", "Generated by": "parameter_robustness_study.ipynb", "Notes": "One-factor-at-a-time robustness figure."},
    {"Report table/figure": "Robustness ratio vs N", "LaTeX filename/path": "parameter_robustness_outputs/robustness_ratio_vs_N.png", "Clean repo path": "figures/report/robustness/robustness_ratio_vs_N.png", "Generated by": "parameter_robustness_study.ipynb", "Notes": "Complements RMSE robustness plot."},
    {"Report table/figure": "Heston full-info RMSE bar", "LaTeX filename/path": "heston_delta_vega_outputs/heston_cos_fullinfo_rmse_bar.png", "Clean repo path": "figures/report/heston_fullinfo/heston_cos_fullinfo_rmse_bar.png", "Generated by": "corrected Heston notebook, USE_OBSERVABLE_VOL=False", "Notes": "Main Heston headline figure."},
    {"Report table/figure": "Heston full-info CVaR95 bar", "LaTeX filename/path": "heston_delta_vega_outputs/heston_cos_fullinfo_cvar95_bar.png", "Clean repo path": "figures/report/heston_fullinfo/heston_cos_fullinfo_cvar95_bar.png", "Generated by": "corrected Heston notebook, USE_OBSERVABLE_VOL=False", "Notes": "CVaR appears here as a risk metric, not a CVaR-trained extension."},
    {"Report table/figure": "Heston option position by variance", "LaTeX filename/path": "heston_delta_vega_outputs/heston_cos_fullinfo_option_position_by_variance.png", "Clean repo path": "figures/report/heston_fullinfo/heston_cos_fullinfo_option_position_by_variance.png", "Generated by": "corrected Heston notebook, USE_OBSERVABLE_VOL=False", "Notes": "Protect full-info mode suffix."},
])
display(latex_mapping)


## Black--Scholes architecture selection and final selected model

Purpose:
Record the selected shared Markov MLP architecture for the controlled Black--Scholes task.

Report use:
Supports the architecture-selection discussion and fixes the model used in later Black--Scholes experiments.

Key caveat:
This section summarizes the selected architecture; it does not rerun the full hyperparameter search by default.

Outputs:
Architecture-selection tables/figures should be copied into `results/report/black_scholes/` and `figures/report/black_scholes/` if retained.


In [ ]:
SELECTED_BS_ARCHITECTURE = pd.DataFrame([
    {"Component": "Input features", "Value": "log(S_t/K), tau/T"},
    {"Component": "Hidden layers", "Value": "3 fully connected layers"},
    {"Component": "Width", "Value": "64"},
    {"Component": "Activation", "Value": "tanh"},
    {"Component": "Output", "Value": "sigmoid delta in (0, 1)"},
    {"Component": "Sharing", "Value": "same feed-forward MLP reused at every hedge date"},
])
display(SELECTED_BS_ARCHITECTURE)
long_run_hook(RUN_TRAINING, "run ideal_minimal_task_hyperparameter_tuning_with_architectures.ipynb for the full architecture search")


## Black--Scholes final benchmark comparison

Purpose:
Compare no hedge, Black--Scholes delta, discrete-time MSE-optimal hedge, polynomial hedge, and neural hedge.

Report use:
Main controlled benchmark table and figures.

Key caveat:
The neural hedge should match analytic/finite-grid references closely rather than beat them in the correctly specified model.

Outputs:
`results/report/black_scholes/final_benchmark_metrics.csv` and Black--Scholes report figures.


In [ ]:
long_run_hook(RUN_TRAINING, "run final_benchmark_comparison.ipynb")
benchmark = display_csv(RESULTS["black_scholes"] / "final_benchmark_metrics.csv", "Final Black--Scholes benchmark table", used_for="main benchmark table", regenerate="run final_benchmark_comparison.ipynb and copy the CSV here")
display_figure(FIGURES["black_scholes"] / "pnl_four_strategies.png", "Four-strategy P&L distribution", used_for="Black--Scholes P&L figure", regenerate="run final_benchmark_comparison.ipynb and copy pnl_four_strategies.png here")
display_figure(FIGURES["black_scholes"] / "pnl_left_tail_zoom.png", "Left-tail P&L zoom", used_for="Black--Scholes tail figure", regenerate="run final_benchmark_comparison.ipynb and copy pnl_left_tail_zoom.png here")
display_figure(FIGURES["black_scholes"] / "average_delta_by_moneyness.png", "Average hedge ratio by moneyness", used_for="delta-shape diagnostic", regenerate="run final_benchmark_comparison.ipynb and copy average_delta_by_moneyness.png here")


## Data-generation comparison

Purpose:
Compare crude Monte Carlo, Latin Hypercube, and Sobol/Sobol Brownian bridge sampling where retained.

Report use:
Supporting report or appendix result.

Key caveat:
Structured sampling benefits should be described as modest at low path budgets and not overclaimed.

Outputs:
`results/report/data_generation/data_generation_comparison.csv` and related figures.


In [ ]:
if INCLUDE_DATA_GENERATION:
    long_run_hook(RUN_TRAINING, "run the data-generation cells in final_benchmark_comparison.ipynb")
    display_csv(RESULTS["data_generation"] / "data_generation_comparison.csv", "Data-generation comparison", used_for="appendix/supporting data-generation table", regenerate="run final_benchmark_comparison.ipynb data-generation cells")
    display_figure(FIGURES["data_generation"] / "data_generation_learning_curves.png", "Data-generation learning curves", used_for="appendix/supporting figure", regenerate="run final_benchmark_comparison.ipynb data-generation cells")
else:
    print("Data-generation comparison excluded by current report configuration.")


## Parameter robustness under retraining

Purpose:
Retrain the selected architecture across one-factor-at-a-time parameter grids.

Report use:
Supports the robustness-under-retraining section.

Key caveat:
This is not out-of-scenario generalization from one fixed baseline network.

Outputs:
`results/report/robustness/robustness_compact.csv` and robustness figures.


In [ ]:
long_run_hook(RUN_TRAINING, "run parameter_robustness_study.ipynb")
display_csv(RESULTS["robustness"] / "robustness_compact.csv", "Compact robustness table", used_for="parameter robustness table", regenerate="run parameter_robustness_study.ipynb")
display_figure(FIGURES["robustness"] / "robustness_rmse_vs_N.png", "Robustness RMSE vs N", used_for="robustness figure", regenerate="run parameter_robustness_study.ipynb")
display_figure(FIGURES["robustness"] / "robustness_ratio_vs_N.png", "Robustness ratio vs N", used_for="robustness figure", regenerate="run parameter_robustness_study.ipynb")


## Generalization failure of the single-scenario hedger

Purpose:
Show that the baseline hedger fails out of scenario without retraining.

Report use:
Motivates the parameter-conditioned hedger.

Key caveat:
The failure is structural because the baseline inputs omit contract variation such as volatility.

Outputs:
`results/report/robustness/collapse_compact.csv`.


In [ ]:
long_run_hook(RUN_TRAINING, "run final_benchmark_comparison.ipynb generalization-failure cells")
display_csv(RESULTS["robustness"] / "collapse_compact.csv", "Single-scenario generalization failure", used_for="generalization failure table", regenerate="run final_benchmark_comparison.ipynb")


## Parameter-conditioned hedger

Purpose:
Train a hedger over a distribution of strikes and volatilities with contract parameters as inputs.

Report use:
Supports interpolation/extrapolation discussion for the conditioned hedger.

Key caveat:
A per-path Black--Scholes premium is supplied because one scalar premium cannot price all contracts.

Outputs:
`results/report/parameter_conditioned/universal_robustness_with_extrapolation.csv` and heatmap/delta figures.


In [ ]:
long_run_hook(RUN_TRAINING, "run final_benchmark_comparison.ipynb parameter-conditioned hedger cells")
display_csv(RESULTS["parameter_conditioned"] / "universal_robustness.csv", "Parameter-conditioned robustness", used_for="conditioned hedger table", regenerate="run final_benchmark_comparison.ipynb")
display_csv(RESULTS["parameter_conditioned"] / "universal_robustness_with_extrapolation.csv", "Parameter-conditioned extrapolation", used_for="conditioned hedger extrapolation table", regenerate="run final_benchmark_comparison.ipynb")
display_figure(FIGURES["parameter_conditioned"] / "universal_heatmap.png", "Universal hedger heatmap", used_for="conditioned hedger figure", regenerate="run final_benchmark_comparison.ipynb")


## Supporting transaction-cost extension

Purpose:
Show how the selected architecture adapts when proportional transaction costs are included.

Report use:
Appendix/supporting result only.

Key caveat:
This is not a full optimal transaction-cost hedging study.

Outputs:
`results/report/transaction_costs/transaction_cost_compact_results.csv` and transaction-cost figures.


In [ ]:
if INCLUDE_TRANSACTION_COSTS:
    long_run_hook(RUN_TRAINING, "run transaction_cost_neural_hedging_extension (1).ipynb")
    display_csv(RESULTS["transaction_costs"] / "transaction_cost_compact_results.csv", "Transaction-cost compact results", used_for="transaction-cost appendix table", regenerate="run transaction_cost_neural_hedging_extension (1).ipynb")
    display_figure(FIGURES["transaction_costs"] / "transaction_cost_rmse_bar_representative.png", "Transaction-cost representative RMSE", used_for="transaction-cost appendix figure", regenerate="run transaction_cost_neural_hedging_extension (1).ipynb")
else:
    print("Transaction-cost extension excluded by current report configuration.")


## Heston COS pricer and Carr--Madan validation

Purpose:
Run a quick validation grid comparing the submission COS pricer with an independent Carr--Madan quadrature.

Report use:
Supports the Heston pricing-validation claim.

Key caveat:
The validation result is grid-specific and should not be read as a global accuracy guarantee over all maturities and variance states.

Outputs:
`results/report/validation/heston_cos_carr_madan_validation_traded_range.csv` and `heston_cos_greek_finite_difference_check.csv`.


In [ ]:
if RUN_QUICK_VALIDATION:
    display(Markdown("### COS/Carr--Madan validation grid"))
    try:
        from src.heston_pricing import finite_difference_greeks, validation_grid
        rows, summary = validation_grid()
        validation_df = pd.DataFrame(list(rows))
        validation_path = RESULTS["validation"] / "heston_cos_carr_madan_validation_traded_range.csv"
        save_dataframe(validation_df, validation_path, index=False, section="Carr--Madan validation", notes="COS/Carr--Madan traded-range grid")
        display(validation_df)
        print(f"Maximum absolute discrepancy: {summary['max_abs_error']:.6e}")
        print(f"Median absolute discrepancy: {summary['median_abs_error']:.6e}")
        print(f"Wrote {validation_path.relative_to(PROJECT_ROOT)}")

        greek_df = pd.DataFrame([finite_difference_greeks()])
        greek_path = RESULTS["validation"] / "heston_cos_greek_finite_difference_check.csv"
        save_dataframe(greek_df, greek_path, index=False, section="Greek finite-difference check", notes="Central finite-difference stability check")
        display(Markdown("### Finite-difference Greek stability check"))
        display(greek_df)
        print(f"Wrote {greek_path.relative_to(PROJECT_ROOT)}")
    except ImportError as exc:
        print("MISSING DEPENDENCY FOR QUICK VALIDATION:")
        print(exc)
        print("Install dependencies with `pip install -r requirements.txt`, then rerun this section.")
else:
    print("Quick validation disabled. Set RUN_QUICK_VALIDATION=True to run the COS/Carr--Madan check.")


## Dual Heston mode generation plan

Purpose:
Make the full-information and observable-volatility Heston runs explicit and mode-safe.

Report use:
The full-information mode is the main Heston result; observable-volatility is a supporting robustness check if retained.

Key caveat:
The two modes must write to separate folders and mode-suffixed filenames. Observable-volatility outputs must never overwrite the full-information headline results.

Outputs:
Mode metadata JSON files under `results/report/heston_fullinfo/` and `results/report/heston_obsvol/`, mirrored to Drive in Colab.


In [ ]:
display(pd.DataFrame(HESTON_MODES)[["name", "use_observable_vol", "description", "results_dir", "figures_dir"]])

for mode in HESTON_MODES:
    write_heston_mode_metadata(mode)

if RUN_HESTON_FULL_GENERATION:
    for mode in HESTON_MODES:
        if mode["name"] == "fullinfo" and not RUN_HESTON_FULLINFO:
            continue
        if mode["name"] == "obsvol" and not RUN_HESTON_OBSVOL:
            continue
        run_heston_experiment(mode["name"], mode["use_observable_vol"])
else:
    print("Full Heston generation is disabled in default review mode. Metadata has still been written for both modes.")


## Heston full-information stock-and-option experiment

Purpose:
Evaluate the two-output neural hedge under Heston dynamics with direct access to simulated variance.

Report use:
Main stochastic-volatility headline experiment.

Key caveat:
Observable-volatility outputs must never replace full-information outputs.

Outputs:
`results/report/heston_fullinfo/heston_revised_results_fair_premium_fullinfo.csv` and mode-safe figures.


In [ ]:
fullinfo_mode = next(m for m in HESTON_MODES if m["name"] == "fullinfo")
write_heston_mode_metadata(fullinfo_mode)
fullinfo_reference = pd.DataFrame([
    {"Strategy": "No hedge", "RMSE": 0.190061},
    {"Strategy": "BS proxy stock delta", "RMSE": 0.019997},
    {"Strategy": "Heston COS delta--vega", "RMSE": 0.014782},
    {"Strategy": "Stock-only NN tanh", "RMSE": 0.018166},
    {"Strategy": "Stock + option NN", "RMSE": 0.007426},
    {"Strategy": "Same-path BS-proxy-Greeks heuristic", "RMSE": 0.009874},
])
display(Markdown("### Protected full-information headline values"))
display(fullinfo_reference)
print("Full-info max |eta| approximately 1.33; multi-seed average improvement approximately 22.7 percent.")
print("Full-information Heston generation is handled by the full-generation dispatcher when RUN_MODE=\"full_generation\".")
display_csv(RESULTS["heston_fullinfo"] / "heston_revised_results_fair_premium_fullinfo.csv", "Heston full-information fair-premium results", used_for="main Heston table", regenerate="run corrected Heston notebook with USE_OBSERVABLE_VOL=False")
display_figure(FIGURES["heston_fullinfo"] / "heston_cos_fullinfo_rmse_bar.png", "Heston full-information RMSE", used_for="main Heston figure", regenerate="run corrected Heston notebook with USE_OBSERVABLE_VOL=False")
display_figure(FIGURES["heston_fullinfo"] / "heston_cos_fullinfo_cvar95_bar.png", "Heston full-information CVaR95 risk metric", used_for="Heston tail-risk metric figure", regenerate="run corrected Heston notebook with USE_OBSERVABLE_VOL=False")
display_figure(FIGURES["heston_fullinfo"] / "heston_cos_fullinfo_option_position_by_variance.png", "Heston option position by variance", used_for="Heston interpretation figure", regenerate="run corrected Heston notebook with USE_OBSERVABLE_VOL=False")


## Heston same-path BS-proxy-Greeks diagnostic

Purpose:
Evaluate the frozen-volatility BS-proxy Greek rule on the same COS-priced Heston option path.

Report use:
Diagnostic heuristic comparison for the Heston section.

Key caveat:
This is not the old proxy-on-proxy headline experiment.

Outputs:
`results/report/heston_fullinfo/heston_same_path_bs_proxy_delta_vega_fair_fullinfo.csv`.


In [ ]:
display_csv(RESULTS["heston_fullinfo"] / "heston_same_path_bs_proxy_delta_vega_fair_fullinfo.csv", "Same-path BS-proxy-Greeks diagnostic", used_for="Heston diagnostic comparison", regenerate="run corrected Heston notebook same-path diagnostic cells")


## Heston full-information multi-seed summary

Purpose:
Check whether the stock-and-option NN improvement is stable across the reported seed check.

Report use:
Supports the conservative multi-seed headline.

Key caveat:
Three seeds are a consistency check, not a high-powered statistical estimate.

Outputs:
`results/report/heston_fullinfo/heston_multiseed_pairwise_improvements_fullinfo.csv`.


In [ ]:
long_run_hook(RUN_LONG_MULTI_SEED, "run corrected Heston notebook multi-seed cell with USE_OBSERVABLE_VOL=False")
display_csv(RESULTS["heston_fullinfo"] / "heston_multiseed_pairwise_improvements_fullinfo.csv", "Heston full-information multi-seed summary", used_for="Heston multi-seed headline", regenerate="run corrected Heston notebook multi-seed cell")


## Observable-volatility Heston robustness check

Purpose:
Repeat the Heston comparison using a causal EWMA variance proxy instead of direct simulated variance.

Report use:
Supporting robustness check if retained in the report/appendix.

Key caveat:
The liquid-option quote is still observable and embeds variance information, so this is not a pure stock-return-only filter.

Outputs:
`results/report/heston_obsvol/heston_revised_results_fair_premium_obsvol.csv` and mode-safe observable-volatility figures.


In [ ]:
if INCLUDE_OBSERVABLE_VOL_HESTON:
    obsvol_mode = next(m for m in HESTON_MODES if m["name"] == "obsvol")
    write_heston_mode_metadata(obsvol_mode)
    OBSERVABLE_VOL_CAVEAT = 'Observable-volatility Heston robustness run\n\nThis run removes direct access to the simulated variance v_t and replaces it with an EWMA proxy v_hat_t. The network still observes the contemporaneous liquid-option quote C_t^h/S_t. That quote is available at the hedging date, so this is not future-information leakage, but it carries implied information about the latent variance state. This run should therefore be interpreted as an observable-market-information robustness check, not as a pure stock-return-only volatility-filtering experiment.\n'
    save_text(OBSERVABLE_VOL_CAVEAT, RESULTS["heston_obsvol"] / "observable_volatility_caveat.txt", section="Observable-volatility caveat")
    obsvol_reference = pd.DataFrame([
        {"Quantity": "Stock + option NN RMSE", "Value": 0.008010},
        {"Quantity": "Same-path BS-proxy-Greeks heuristic RMSE", "Value": 0.010725},
        {"Quantity": "Multi-seed mean NN RMSE", "Value": 0.007877},
        {"Quantity": "Multi-seed mean heuristic RMSE", "Value": 0.010734},
        {"Quantity": "Average RMSE improvement", "Value": "approximately 26.6 percent"},
        {"Quantity": "max |eta|", "Value": "approximately 1.13"},
    ])
    display(Markdown("### Protected observable-volatility reference values"))
    display(obsvol_reference)
    print("Observable-volatility Heston generation is handled by the full-generation dispatcher when RUN_MODE=\"full_generation\".")
    display_csv(RESULTS["heston_obsvol"] / "heston_revised_results_fair_premium_obsvol.csv", "Observable-volatility Heston results", used_for="observable-volatility robustness table", regenerate="run corrected Heston notebook with USE_OBSERVABLE_VOL=True")
    display_csv(RESULTS["heston_obsvol"] / "heston_multiseed_pairwise_improvements_obsvol.csv", "Observable-volatility multi-seed summary", used_for="observable-volatility robustness headline", regenerate="run corrected Heston notebook multi-seed cell with USE_OBSERVABLE_VOL=True")
    display_figure(FIGURES["heston_obsvol"] / "heston_cos_obsvol_rmse_bar.png", "Observable-volatility Heston RMSE", used_for="observable-volatility robustness figure", regenerate="run corrected Heston notebook with USE_OBSERVABLE_VOL=True")
else:
    print("Observable-volatility Heston excluded by current report configuration.")


## Excluded CVaR-trained loss extension

Purpose:
Document the explicit exclusion of the removed CVaR-trained loss workflow.

Report use:
None unless the final report restores this appendix.

Key caveat:
CVaR may appear as a risk metric such as Loss CVaR95, but this notebook does not reproduce a CVaR-trained extension.

Outputs:
No report outputs.


In [ ]:
if INCLUDE_CVAR_TRAINED_LOSS:
    raise RuntimeError("CVaR-trained loss is disabled unless the final report explicitly restores it.")
print("CVaR-trained loss extension excluded. CVaR is used only as a risk metric where it appears in result tables/figures.")


## Final output manifest and missing-output summary

Purpose:
Write a CSV manifest showing expected outputs, report use, clean paths, and availability.

Report use:
Gives the group a final checklist before submission.

Key caveat:
Missing outputs must be regenerated or copied from verified runs; dummy rows are not acceptable.

Outputs:
`results/report/report_output_manifest.csv`.


In [ ]:
manifest = latex_mapping.rename(columns={"Clean repo path": "local_path", "Report table/figure": "report_table_or_figure"}).copy()
manifest["experiment"] = manifest["report_table_or_figure"].str.extract(r"^(Black--Scholes|Heston|Robustness|Parameter|Final|Four|Left|Average)", expand=False).fillna("Report")
manifest["mode"] = manifest["local_path"].apply(lambda p: "fullinfo" if "fullinfo" in p else ("obsvol" if "obsvol" in p else ""))
manifest["output_type"] = manifest["local_path"].map(lambda p: Path(p).suffix.lstrip("."))
manifest["used_in_report"] = "yes"
manifest["timestamp"] = now_utc()
manifest["drive_path"] = manifest["local_path"].map(lambda p: str(drive_path_for(PROJECT_ROOT / p)) if drive_path_for(PROJECT_ROOT / p) else "")

extra = pd.DataFrame([
    {"experiment": "Transaction costs", "mode": "", "output_type": "csv", "local_path": "results/report/transaction_costs/transaction_cost_compact_results.csv", "drive_path": str(drive_path_for(PROJECT_ROOT / "results/report/transaction_costs/transaction_cost_compact_results.csv") or ""), "used_in_report": "if retained", "report_table_or_figure": "Transaction-cost compact table", "Generated by": "transaction_cost_neural_hedging_extension (1).ipynb", "Notes": "Supporting appendix only.", "timestamp": now_utc()},
    {"experiment": "Heston", "mode": "obsvol", "output_type": "csv", "local_path": "results/report/heston_obsvol/heston_revised_results_fair_premium_obsvol.csv", "drive_path": str(drive_path_for(PROJECT_ROOT / "results/report/heston_obsvol/heston_revised_results_fair_premium_obsvol.csv") or ""), "used_in_report": "if retained", "report_table_or_figure": "Observable-volatility Heston results", "Generated by": "corrected Heston notebook, USE_OBSERVABLE_VOL=True", "Notes": "Supporting robustness only.", "timestamp": now_utc()},
    {"experiment": "Heston validation", "mode": "", "output_type": "csv", "local_path": "results/report/validation/heston_cos_carr_madan_validation_traded_range.csv", "drive_path": str(drive_path_for(PROJECT_ROOT / "results/report/validation/heston_cos_carr_madan_validation_traded_range.csv") or ""), "used_in_report": "yes", "report_table_or_figure": "COS/Carr--Madan validation", "Generated by": "this notebook", "Notes": "Quick validation code runs in review mode.", "timestamp": now_utc()},
    {"experiment": "Heston validation", "mode": "", "output_type": "csv", "local_path": "results/report/validation/heston_cos_greek_finite_difference_check.csv", "drive_path": str(drive_path_for(PROJECT_ROOT / "results/report/validation/heston_cos_greek_finite_difference_check.csv") or ""), "used_in_report": "yes", "report_table_or_figure": "Greek finite-difference check", "Generated by": "this notebook", "Notes": "Quick finite-difference stability check.", "timestamp": now_utc()},
])

keep_cols = ["experiment", "mode", "output_type", "local_path", "drive_path", "used_in_report", "report_table_or_figure", "timestamp", "generated_in_current_run", "Generated by", "Notes"]
manifest = pd.concat([manifest, extra], ignore_index=True)
for col in keep_cols:
    if col not in manifest.columns:
        manifest[col] = ""
manifest = manifest[keep_cols]
manifest["generated_in_current_run"] = manifest["local_path"].map(lambda p: str(p) in generated_in_current_run)
manifest["Present"] = manifest["local_path"].map(lambda p: (PROJECT_ROOT / p).exists())
manifest_path = RESULTS_ROOT / "report_output_manifest.csv"
save_dataframe(manifest, manifest_path, index=False, section="Final output manifest", notes="local and Drive paths included")
display(manifest)
missing = manifest.loc[~manifest["Present"], ["local_path", "report_table_or_figure", "Generated by"]]
if len(missing):
    display(Markdown("### Missing required outputs"))
    display(missing)
else:
    print("All expected report outputs are present.")


## Google Drive zip export

Purpose:
Package the cleaned report outputs, manifest, documentation, and submission notebook into one archive.

Report use:
Provides a portable bundle for upload or sharing after a Colab run.

Key caveat:
The zip contains whatever outputs exist at the time it is run. If required CSVs or figures are missing, rerun/copy them before treating the zip as final.

Outputs:
`report_outputs_latest.zip` locally and, in Colab with Drive enabled, `MyDrive/MFE_neural_hedging_report_outputs/zips/report_outputs_latest.zip`.


In [ ]:
zip_items = [
    RESULTS_ROOT,
    FIGURES_ROOT,
    PROJECT_ROOT / "docs" / "results_manifest.md",
    PROJECT_ROOT / "notebooks" / "01_reproduce_report_results.ipynb",
]
local_zip_path = PROJECT_ROOT / "report_outputs_latest.zip"
if local_zip_path.exists():
    local_zip_path.unlink()
with zipfile.ZipFile(local_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in zip_items:
        item = Path(item)
        if item.is_dir():
            for child in item.rglob("*"):
                if child.is_file():
                    zf.write(child, child.relative_to(PROJECT_ROOT))
        elif item.exists():
            zf.write(item, item.relative_to(PROJECT_ROOT))
print(f"Saved local zip: {local_zip_path}")
zip_drive_path = None
if DRIVE_ROOT is not None:
    zip_drive_path = DRIVE_ROOT / "zips" / "report_outputs_latest.zip"
    zip_drive_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(local_zip_path, zip_drive_path)
    print(f"Saved Drive zip: {zip_drive_path}")
log_progress("Google Drive zip export", "saved", local_zip_path, zip_drive_path, "report output bundle")

if IN_COLAB and DOWNLOAD_ZIP_AT_END:
    from google.colab import files  # type: ignore
    files.download(str(local_zip_path))
